In [1]:
# STEP ONE

import pandas as pd

df = pd.read_csv(
    "customs_clean.csv",
    parse_dates=["entry_date", "accepted_date", "release_date"],
    dtype={"hs_code": str},
)

In [2]:
df.columns

Index(['shipment_id', 'entry_no', 'hs_code', 'product_desc', 'supplier',
       'country_origin', 'country_destination', 'declared_value_usd',
       'invoice_value_usd', 'weight_kg', 'volume_m3', 'broker',
       'port_of_entry', 'entry_date', 'accepted_date', 'release_date',
       'duty_amount_usd', 'tax_amount_usd', 'broker_fee_usd', 'incoterm',
       'license_required', 'license_obtained', 'fta_claimed',
       'restricted_party_pass', 'container_number', 'slow_clearance',
       'hs_chapter', 'hs_section'],
      dtype='str')

In [3]:
# STEP TWO

# Build three new columns:
# Column Meanings:-
    # entry_day_of_week	(Day of the week as a number or name)
    # entry_month	(Month of the year)
    # entry_is_weekend	True or False

In [4]:
df['entry_date'].head(3)

0   2026-07-29 23:35:56
1   2025-11-06 23:35:56
2   2026-06-20 23:35:56
Name: entry_date, dtype: datetime64[us]

In [5]:
# Creating a column for day of the week, month and weekend
df["entry_day_of_week"] = df["entry_date"].dt.dayofweek    # 0 = Monday ... 6 = Sunday
df["entry_month"] = df["entry_date"].dt.month              # 1 to 12
df["entry_is_weekend"] = df["entry_day_of_week"] >= 5      # Saturday (5) and Sunday (6)

In [6]:
df["entry_day_of_week"].value_counts().sort_index()

entry_day_of_week
0    684
1    683
2    676
3    685
4    728
5    637
6    653
Name: count, dtype: int64

In [7]:
df.groupby("entry_day_of_week")["slow_clearance"].value_counts(normalize=True)

entry_day_of_week  slow_clearance
0                  N                 0.730994
                   Y                 0.269006
1                  N                 0.739385
                   Y                 0.260615
2                  N                 0.772189
                   Y                 0.227811
3                  N                 0.775182
                   Y                 0.224818
4                  N                 0.758242
                   Y                 0.241758
5                  N                 0.690738
                   Y                 0.309262
6                  N                 0.702910
                   Y                 0.297090
Name: proportion, dtype: float64

In [8]:
df["entry_month"].value_counts().sort_index()

entry_month
1     417
2     364
3     369
4     405
5     399
6     403
7     404
8     387
9     404
10    424
11    390
12    380
Name: count, dtype: int64

In [9]:
df["entry_is_weekend"].value_counts()

entry_is_weekend
False    3456
True     1290
Name: count, dtype: int64

In [10]:
df["entry_month"].value_counts().sort_index()

entry_month
1     417
2     364
3     369
4     405
5     399
6     403
7     404
8     387
9     404
10    424
11    390
12    380
Name: count, dtype: int64

In [11]:
df.groupby("entry_month")["slow_clearance"].value_counts(normalize=True)

entry_month  slow_clearance
1            N                 0.726619
             Y                 0.273381
2            N                 0.708791
             Y                 0.291209
3            N                 0.756098
             Y                 0.243902
4            N                 0.716049
             Y                 0.283951
5            N                 0.719298
             Y                 0.280702
6            N                 0.771712
             Y                 0.228288
7            N                 0.772277
             Y                 0.227723
8            N                 0.733850
             Y                 0.266150
9            N                 0.764851
             Y                 0.235149
10           N                 0.745283
             Y                 0.254717
11           N                 0.741026
             Y                 0.258974
12           N                 0.713158
             Y                 0.286842
Name: propor

In [12]:
df.groupby("entry_is_weekend")["slow_clearance"].value_counts(normalize=True)

entry_is_weekend  slow_clearance
False             N                 0.755208
                  Y                 0.244792
True              N                 0.696899
                  Y                 0.303101
Name: proportion, dtype: float64

In [13]:
df['license_obtained'].value_counts()

license_obtained
N    4624
Y     122
Name: count, dtype: int64

In [14]:
df['license_required'].value_counts()

license_required
N    4574
Y     172
Name: count, dtype: int64

In [15]:
def check_license(row):
    if (row['license_required'] == 'Y') and (row['license_obtained'] == 'N'):
        return 'Y'
    else:
        return 'N'

df["license_missing"] = df.apply(check_license, axis=1)

In [16]:
df["license_missing"].value_counts()

license_missing
N    4696
Y      50
Name: count, dtype: int64

In [17]:
df.groupby("license_missing")["slow_clearance"].value_counts()

license_missing  slow_clearance
N                N                 3483
                 Y                 1213
Y                N                   26
                 Y                   24
Name: count, dtype: int64

In [18]:
df.groupby("license_missing")["slow_clearance"].value_counts(normalize=True)

license_missing  slow_clearance
N                N                 0.741695
                 Y                 0.258305
Y                N                 0.520000
                 Y                 0.480000
Name: proportion, dtype: float64

In [19]:
df.columns

Index(['shipment_id', 'entry_no', 'hs_code', 'product_desc', 'supplier',
       'country_origin', 'country_destination', 'declared_value_usd',
       'invoice_value_usd', 'weight_kg', 'volume_m3', 'broker',
       'port_of_entry', 'entry_date', 'accepted_date', 'release_date',
       'duty_amount_usd', 'tax_amount_usd', 'broker_fee_usd', 'incoterm',
       'license_required', 'license_obtained', 'fta_claimed',
       'restricted_party_pass', 'container_number', 'slow_clearance',
       'hs_chapter', 'hs_section', 'entry_day_of_week', 'entry_month',
       'entry_is_weekend', 'license_missing'],
      dtype='str')

In [20]:
# CHECKING DISCREPANCIES BETWEEN DECLARED VALUE AND INVOICE VALUES
df['gap_pct'] = (df['invoice_value_usd'] - df['declared_value_usd'])/ df['invoice_value_usd']


In [21]:
df["gap_pct"].describe()

count    4746.000000
mean        0.054666
std         0.126095
min        -0.052625
25%        -0.019677
50%         0.011454
75%         0.040074
max         0.500000
Name: gap_pct, dtype: float64

In [22]:
df["gap_pct"].quantile([0.5, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0])

0.50    0.011454
0.75    0.040074
0.80    0.045486
0.85    0.231176
0.90    0.273701
0.95    0.317264
1.00    0.500000
Name: gap_pct, dtype: float64

In [23]:
pd.cut(df["gap_pct"], bins=[-1, -0.05, 0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.6]).value_counts().sort_index()

gap_pct
(-1.0, -0.05]      87
(-0.05, 0.0]     1853
(0.0, 0.05]      1950
(0.05, 0.1]         0
(0.1, 0.15]         0
(0.15, 0.2]         0
(0.2, 0.3]        517
(0.3, 0.4]        210
(0.4, 0.6]        129
Name: count, dtype: int64

In [24]:
df["high_discrepancy"] = (df["gap_pct"] > 0.10).map({True: 'Y', False: 'N'})
df["high_discrepancy"].value_counts()

high_discrepancy
N    3890
Y     856
Name: count, dtype: int64

In [25]:
df["high_discrepancy"].value_counts(normalize=True)

high_discrepancy
N    0.819638
Y    0.180362
Name: proportion, dtype: float64

In [26]:
for t in [0.05, 0.10, 0.15, 0.20]:
    print(t, (df["gap_pct"] > t).sum())

0.05 856
0.1 856
0.15 856
0.2 856


In [27]:
lookup = pd.DataFrame({
    "hs_section": ["VI", "VII", "XI", "XIII", "XV", "XVI", "XVII", "XX"],
    "hs_section_name": [
        "Chemical and pharmaceutical products",
        "Plastics and rubber",
        "Textiles",
        "Stone, ceramic and glass",
        "Base metals",
        "Machinery and electrical",
        "Vehicles and transport equipment",
        "Miscellaneous manufactured articles",
    ],
})

df = df.merge(lookup, on="hs_section", how="left")

print(df.shape)                                   # expect (4746, 35)
print(df["hs_section_name"].isnull().sum())       # expect 0

(4746, 35)
0


In [28]:
df["slow_flag"] = df["slow_clearance"] == "Y"
df["discrepancy_flag"] = df["high_discrepancy"] == "Y"

In [29]:
def summary(col):
    return df.groupby(col).agg(
        rows=("shipment_id", "count"),
        slow_rate=("slow_flag", "mean"),
        discrepancy_rate=("discrepancy_flag", "mean"),
    ).sort_values("slow_rate", ascending=False).round(3)

port_table = summary("port_of_entry")
broker_table = summary("broker")
section_table = summary("hs_section_name")
weekend_table = summary("entry_is_weekend")
license_table = summary("license_missing")
origin_table = summary("country_origin")

port_table

,rows,slow_rate,discrepancy_rate
port_of_entry,,,
ALEXANDRIA,701,0.335,0.185
MOMBASA,719,0.307,0.171
LAGOS,701,0.307,0.180
CASABLANCA,743,0.301,0.176
MOMBASA ICD,231,0.208,0.234
DURBAN,255,0.196,0.169
TEMA,258,0.194,0.163
DAR ES SALAAM,227,0.189,0.141
PORT ELIZABETH,239,0.188,0.188


In [30]:
df = df.drop(columns=["slow_flag", "discrepancy_flag"])
df.to_csv("customs_features.csv", index=False)
print(df.shape)                                   # expect (4746, 35)

(4746, 35)


In [31]:
check = pd.read_csv(
    "customs_features.csv",
    parse_dates=["entry_date", "accepted_date", "release_date"],
    dtype={"hs_code": str},
)
print(check.shape)                                # (4746, 35)
check[["entry_is_weekend", "license_missing", "high_discrepancy", "hs_section_name"]].head()

(4746, 35)


,entry_is_weekend,license_missing,high_discrepancy,hs_section_name
0,False,N,N,Chemical and pharmaceutical products
1,False,N,N,Machinery and electrical
2,True,N,N,Chemical and pharmaceutical products
3,False,N,N,Plastics and rubber
4,False,N,N,Vehicles and transport equipment
